# P2 · N4 — Downstream Impact, Fairness, and Cost

**Paper 2 — MARQ-Bench: Machine-Authored Data Quality**

Answers RQ5: does gating training data with machine-authored rules change
downstream model discrimination or subgroup fairness, and at what cost?

**Design.** Gating is a training-data curation decision, so the gate is applied
to the **training partition only**. Every condition is then evaluated on the
**same ungated test partition** — otherwise each condition would be scored on a
different population and the comparison would be meaningless.

```
evaluation split (80%, from N0)
  |-- train 70%   <- gate applied here
  |-- test  30%   <- never gated, identical for every condition
```

**Infeasible gates are a result.** A rule set that empties the training
partition, or removes an entire class, makes fitting impossible. That is
recorded with a reason, never skipped — roughly a sixth of rule sets fall into
this category and dropping them would flatter the rest.

**Prerequisites:** N0–N3 complete.
**Runtime:** 20–40 minutes. Checkpointed; safe to interrupt. No API calls.

## 1 · Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Setup

In [ ]:
import sys, json, datetime
from pathlib import Path
import pandas as pd, numpy as np

ROOT        = Path('/content/drive/MyDrive/Paper2_RuleAuthorship')
MODULES     = ROOT / 'Notebooks'
CHECKPOINTS = ROOT / 'checkpoints'
ARTIFACTS   = ROOT / 'artifacts'
RUNS        = ROOT / 'runs'

assert ROOT.exists() and MODULES.exists()
ARTIFACTS.mkdir(parents=True, exist_ok=True)
if str(MODULES) not in sys.path:
    sys.path.insert(0, str(MODULES))

import llmauth_census as C
import llmauth_ir as IR
import llmauth_downstream as DS
import llmauth_checkpoint as CK

ckpt = CK.Checkpoint(CHECKPOINTS)
RUN_LOG_PATH = RUNS / 'runs.jsonl'
EXCLUDE_MODELS = {'mock-1', 'qwen2.5-7b', 'claude-opus-4-8'}

PATTERNS = {
    'bank_marketing':   ['bankfull', 'bank-full', 'bank_full'],
    'diabetes_130us':   ['diabetic_data', 'diabetes'],
    'online_retail_ii': ['online_retail', 'online retail', 'retail'],
    'nyc_tlc_yellow':   ['yellow_tripdata'],
}

def discover_data():
    files = [p for p in ROOT.rglob('*')
             if p.is_file() and p.suffix.lower() in ('.csv','.parquet','.xlsx')]
    out = {}
    for corpus, frags in PATTERNS.items():
        hits = [p for p in files if any(f in p.name.lower() for f in frags)]
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
        if hits: out[corpus] = hits[0]
    return out

print('census', C.CENSUS_VERSION, '| ir', IR.SCHEMA_VERSION,
      '| downstream', DS.DOWNSTREAM_VERSION)
print('supervised tasks:', list(DS.TASKS))

census 1.0.0 | ir 1.0.0 | downstream 1.0.0
supervised tasks: ['bank_marketing', 'diabetes_130us', 'nyc_tlc_yellow']


## 3 · Tasks and leakage exclusions

Each corpus has features that encode the label. These are excluded by name, and
the exclusions belong in the paper's Methods.

`online_retail_ii` has no natural supervised label and is not analysed here; it
contributed detection, retention, and cost only.

In [ ]:
for cid, t in DS.TASKS.items():
    print(f'{cid}')
    print(f'   target    : {t.target_column}  (positive = '
          f'{"callable" if callable(t.positive) else repr(t.positive)})')
    print(f'   excluded  : {t.drop_columns}')
    print(f'   subgroups : {t.subgroups}')
    if t.max_rows: print(f'   subsample : {t.max_rows:,} rows')
    print(f'   note      : {t.note}\n')

bank_marketing
   target    : y  (positive = 'yes')
   excluded  : ['y', 'duration']
   subgroups : ['age_band', 'job', 'marital']
   note      : `duration` dropped: known only after the call, documented by UCI as unsuitable for realistic prediction.

diabetes_130us
   target    : readmitted  (positive = '<30')
   excluded  : ['readmitted', 'encounter_id', 'patient_nbr']
   subgroups : ['race', 'gender', 'age']
   note      : Binarised to readmission within 30 days versus not.

nyc_tlc_yellow
   target    : tip_amount  (positive = callable)
   excluded  : ['tip_amount', 'total_amount', 'payment_type']
   subgroups : ['payment_type', 'PULocationID_band']
   subsample : 250,000 rows
   note      : Label is a RECORDED tip: cash tips are not captured. `total_amount` includes the tip and `payment_type` determines whether one is recorded, so both are excluded as leakage.



## 4 · Load rule sets from the run log

In [ ]:
records = [json.loads(l) for l in open(RUN_LOG_PATH) if l.strip()]
records = [r for r in records if r['key']['model_id'] not in EXCLUDE_MODELS]
records = [r for r in records if r['key']['corpus_id'] in DS.TASKS]

rulesets = {}
for r in records:
    k = r['key']
    key = (k['corpus_id'], k['condition'], k['model_id'], k['seed'])
    rulesets[key] = IR.parse_llm_response(
        r['raw_response'] or '', corpus_id=k['corpus_id'],
        condition=k['condition'], model_id=k['model_id'], seed=k['seed'])

print(f'{len(rulesets)} rule sets across {len(DS.TASKS)} supervised corpora')
print(pd.Series([k[0] for k in rulesets]).value_counts().to_string())

360 rule sets across 3 supervised corpora
bank_marketing    120
diabetes_130us    120
nyc_tlc_yellow    120


## 5 · Build features and the train/test partition

Encoding is fitted on the full evaluation split **before** any gating, so every
condition sees an identical feature space. Encoding uses no label information,
so this does not leak.

In [ ]:
DATA_PATHS = discover_data()
prepared = {}

for cid, task in DS.TASKS.items():
    df, _ = C.load_corpus(cid, DATA_PATHS[cid])
    split = ckpt.step(f'split_{cid}', 'json',
                      lambda: (_ for _ in ()).throw(RuntimeError('run N0 first')))[0]
    ev = df.loc[split['evaluation_index']]

    # subsample ONCE so masks and features refer to the same rows
    if task.max_rows and len(ev) > task.max_rows:
        ev = ev.sample(n=task.max_rows, random_state=DS.SPLIT_SEED).sort_index()

    X, y, strata = DS.prepare_features(ev, task)
    tr, te = DS.train_test_split_eval(ev.index)
    prepared[cid] = dict(ev=ev, X=X, y=y, strata=strata, train=tr, test=te)

    print(f'{cid:<18} eval {len(ev):>7,}  train {len(tr):>7,}  test {len(te):>7,}  '
          f'features {X.shape[1]:>2}  positive {y.mean():.4f}')

  [cached] split_bank_marketing.json  (written 2026-08-08T03:39:10+00:00)
bank_marketing     eval  36,169  train  25,318  test  10,851  features 15  positive 0.1171
  [cached] split_diabetes_130us.json  (written 2026-08-08T03:39:11+00:00)
diabetes_130us     eval  81,413  train  56,989  test  24,424  features 47  positive 0.1120
  [cached] split_nyc_tlc_yellow.json  (written 2026-08-08T03:39:14+00:00)
nyc_tlc_yellow     eval 250,000  train 175,000  test  75,000  features 17  positive 0.6204


## 6 · No-gate baselines

Every gated configuration is compared against this. If no gate beats it, that
is the finding.

In [ ]:
def _baselines():
    out = []
    for cid, p in prepared.items():
        r = DS.fit_and_evaluate(p['X'], p['y'], p['strata'], p['train'], p['test'],
                                None, corpus_id=cid, condition='NO_GATE',
                                model_id='baseline', seed=0)
        out.append(r.to_dict())
    return out

baselines, _ = ckpt.step('n4_baselines', 'json', _baselines,
                         code_version=DS.DOWNSTREAM_VERSION)
base_by_corpus = {b['corpus_id']: b for b in baselines}

print(f'{"corpus":<18}{"ROC-AUC":>9}{"PR-AUC":>9}{"Brier":>9}{"train rows":>12}')
print('-'*57)
for b in baselines:
    print(f'{b["corpus_id"]:<18}{b["roc_auc"]:>9.4f}{b["pr_auc"]:>9.4f}'
          f'{b["brier"]:>9.4f}{b["train_rows"]:>12,}')

  [cached] n4_baselines.json  (written 2026-08-09T02:44:42+00:00)
corpus              ROC-AUC   PR-AUC    Brier  train rows
---------------------------------------------------------
bank_marketing       0.7900   0.4367   0.0845      25,318
diabetes_130us       0.6676   0.2142   0.0942      56,989
nyc_tlc_yellow       0.8687   0.8773   0.1124     175,000


## 7 · Evaluate every gate

Masks are cached by canonical rule form, so a rule recurring across seeds and
conditions is executed once. Checkpointed — safe to interrupt and re-run.

In [ ]:
mask_cache = {}

def rule_mask(cid, rule):
    key = (cid, IR.canonical_form(rule))
    if key not in mask_cache:
        try:
            mask_cache[key] = IR.to_pandas_mask(rule)(prepared[cid]['ev'])
        except Exception:
            mask_cache[key] = None
    return mask_cache[key]

def gate_mask(cid, rs):
    ev = prepared[cid]['ev']
    keep = pd.Series(True, index=ev.index)
    for rule in rs.rules:
        if not rule.is_executable or rule.column not in ev.columns:
            continue
        m = rule_mask(cid, rule)
        if m is not None:
            keep &= m
    return keep

def _run_all():
    out = []
    items = sorted(rulesets.items())
    for i, (key, rs) in enumerate(items, 1):
        cid, cond, model, seed = key
        p = prepared[cid]
        r = DS.fit_and_evaluate(p['X'], p['y'], p['strata'], p['train'], p['test'],
                                gate_mask(cid, rs), corpus_id=cid, condition=cond,
                                model_id=model, seed=seed)
        out.append(r.to_dict())
        if i % 20 == 0 or i == len(items):
            nf = sum(1 for o in out if not o['feasible'])
            print(f'  {i}/{len(items)}   infeasible so far: {nf}')
    return out

# force=True ensures we re-process and pick up the new Gemini records
results, cached = ckpt.step('n4_downstream_results', 'json', _run_all,
                            code_version=DS.DOWNSTREAM_VERSION, force=True)
res = pd.DataFrame(results)
print(f'\n{len(res)} gates evaluated, {int((~res.feasible).sum())} infeasible')

  [build ] n4_downstream_results.json ...
  20/360   infeasible so far: 2
  40/360   infeasible so far: 3
  60/360   infeasible so far: 10
  80/360   infeasible so far: 14
  100/360   infeasible so far: 31
  120/360   infeasible so far: 45
  140/360   infeasible so far: 55
  160/360   infeasible so far: 55
  180/360   infeasible so far: 55
  200/360   infeasible so far: 55
  220/360   infeasible so far: 55
  240/360   infeasible so far: 55
  260/360   infeasible so far: 60
  280/360   infeasible so far: 60
  300/360   infeasible so far: 60
  320/360   infeasible so far: 60
  340/360   infeasible so far: 60
  360/360   infeasible so far: 60
  [saved ] n4_downstream_results.json  (438,716 bytes)

360 gates evaluated, 60 infeasible


## 8 · Infeasible gates

A gate that cannot be fitted is a governance failure with a concrete
consequence: the data product cannot be built.

In [ ]:
inf = res[~res.feasible]
print(f'{len(inf)}/{len(res)} gates ({len(inf)/len(res):.1%}) make training impossible\n')
if len(inf):
    print(inf.pivot_table(index='corpus_id', columns='condition',
                          values='seed', aggfunc='count').fillna(0).astype(int).to_string())
    print('\nreasons:')
    print(inf.reason.value_counts().head(8).to_string())

60/360 gates (16.7%) make training impossible

condition       A2  A3  A4  A5
corpus_id                     
bank_marketing   2   8  13  22
diabetes_130us  10   0   0   0
nyc_tlc_yellow   5   0   0   0

reasons:
reason
gate left 0 training rows (< 500)                      32
gate left 10 training rows (< 500)                     22
gate left 5 training rows (< 500)                       1
gate left 0 positive / 6012 negative training rows      1
gate left 26 training rows (< 500)                      1
gate left 34 training rows (< 500)                      1
gate left 0 positive / 6020 negative training rows      1
gate left 0 positive / 21220 negative training rows     1


## 9 · Downstream discrimination versus no gate

The headline comparison. `delta_auc` is the gated model's ROC-AUC minus the
no-gate baseline on the same test set. Positive means the gate helped.

In [ ]:
feas = res[res.feasible].copy()
feas['base_auc'] = feas.corpus_id.map(lambda c: base_by_corpus[c]['roc_auc'])
feas['delta_auc'] = feas.roc_auc - feas.base_auc

piv = feas.pivot_table(index=['corpus_id','model_id'], columns='condition',
                       values='delta_auc', aggfunc='mean').round(4)
print('DELTA ROC-AUC vs NO GATE\n'); print(piv.to_string())

print('\n\npooled by condition:')
print(feas.groupby('condition').delta_auc.agg(['mean','std','min','max']).round(4).to_string())

def boot_ci(x, n=10000):
    x = np.asarray(x)
    bs = [np.random.choice(x, len(x)).mean() for _ in range(n)]
    return np.percentile(bs, [2.5, 97.5])

print('\n95% bootstrap CI on mean delta_auc, by condition:')
for c in sorted(feas.condition.unique()):
    x = feas[feas.condition==c].delta_auc.values
    lo, hi = boot_ci(x)
    verdict = 'improves' if lo > 0 else ('degrades' if hi < 0 else 'no effect')
    print(f'  {c}: {x.mean():+.4f}  CI [{lo:+.4f}, {hi:+.4f}]   {verdict}')

best = feas.delta_auc.max()
print(f'\nbest single gate: {best:+.4f} AUC vs baseline')
print(f'gates that beat no-gate at all: {(feas.delta_auc>0).sum()}/{len(feas)} '
      f'({(feas.delta_auc>0).mean():.1%})')

DELTA ROC-AUC vs NO GATE

condition                                      A2      A3      A4      A5
corpus_id      model_id                                                  
bank_marketing claude-haiku-4-5-20251001  -0.0156  0.0010 -0.0053 -0.0115
               claude-sonnet-4-5-20250929 -0.0153 -0.0186 -0.0257 -0.0652
               gemini-flash               -0.0233 -0.0665 -0.1111 -0.1111
diabetes_130us claude-haiku-4-5-20251001      NaN -0.0266 -0.0003 -0.0018
               claude-sonnet-4-5-20250929 -0.0208 -0.0375 -0.0042 -0.0004
               gemini-flash                0.0001 -0.0013 -0.0021 -0.0007
nyc_tlc_yellow claude-haiku-4-5-20251001  -0.3271 -0.3465 -0.2495 -0.2404
               claude-sonnet-4-5-20250929 -0.2891 -0.3647 -0.2814 -0.2164
               gemini-flash               -0.3007 -0.2261 -0.3028 -0.2214


pooled by condition:
             mean     std     min     max
condition                                
A2        -0.1136  0.1441 -0.3707  0.0035
A3        -

## 10 · Subgroup fairness

Two measures. `auc_gap` is the spread in discrimination across levels of a
protected attribute, on the shared test set. `representation_shift` is how much
the gate changed the composition of the training population — half the total
variation distance between the subgroup distribution before and after gating.

In [ ]:
sub = DS.subgroup_report([DS.DownstreamResult(**{
    k: v for k, v in r.items() if k in DS.DownstreamResult.__dataclass_fields__})
    for r in results])
sub = sub[sub.feasible]

print('MEAN SUBGROUP AUC GAP\n')
print(sub.pivot_table(index=['corpus','subgroup'], columns='condition',
                      values='auc_gap', aggfunc='mean').round(4).to_string())

print('\n\nMEAN REPRESENTATION SHIFT (0 = gate did not change composition)\n')
print(sub.pivot_table(index=['corpus','subgroup'], columns='condition',
                      values='representation_shift', aggfunc='mean').round(4).to_string())

worst = sub.sort_values('representation_shift', ascending=False).head(10)
print('\n\nlargest composition shifts:')
print(worst[['corpus','subgroup','condition','model','representation_shift',
             'auc_gap']].to_string(index=False))

MEAN SUBGROUP AUC GAP

condition                             A2      A3      A4      A5
corpus         subgroup                                         
bank_marketing age_band           0.1428  0.1237  0.1315  0.1362
               job                0.1509  0.1696  0.1567  0.1929
               marital            0.0502  0.0520  0.0467  0.0463
diabetes_130us age                0.1832  0.1970  0.1796  0.1791
               gender             0.0130  0.0160  0.0150  0.0134
               race               0.1888  0.1819  0.1773  0.1732
nyc_tlc_yellow PULocationID_band  0.1138  0.1271  0.1126  0.1167
               payment_type       0.3413  0.3386  0.3187  0.2904


MEAN REPRESENTATION SHIFT (0 = gate did not change composition)

condition                             A2      A3      A4      A5
corpus         subgroup                                         
bank_marketing age_band           0.0095  0.0134  0.0117  0.0301
               job                0.0112  0.0137  0.0119  0.0305


## 11 · Cost

Authoring cost alone understates the cost of a bad gate. Cost per million
retained training records charges a rule set for the data it destroys.

In [ ]:
costs = {}
for r in records:
    k = r['key']
    costs[(k['corpus_id'], k['condition'], k['model_id'], k['seed'])] = {
        'cost_usd': r.get('cost_usd') or 0.0,
        'input_tokens': r.get('input_tokens') or 0,
        'output_tokens': r.get('output_tokens') or 0,
        'latency_seconds': r.get('latency_seconds') or 0.0,
    }

feas['cost_usd'] = feas.apply(
    lambda r: costs.get((r.corpus_id, r.condition, r.model_id, r.seed), {}).get('cost_usd', 0.0),
    axis=1)
feas['cost_per_M_retained'] = feas.apply(
    lambda r: DS.cost_per_retained_record(r.cost_usd, r.train_rows), axis=1)

print('MEAN AUTHORING COST (USD per rule set)\n')
print(feas.pivot_table(index='model_id', columns='condition',
                       values='cost_usd', aggfunc='mean').round(4).to_string())

print('\n\nMEAN COST PER MILLION RETAINED TRAINING RECORDS (USD)\n')
print(feas.pivot_table(index=['corpus_id','model_id'], columns='condition',
                       values='cost_per_M_retained', aggfunc='mean').round(3).to_string())

tot = sum(c['cost_usd'] for c in costs.values())
print(f'\ntotal authoring cost across all runs: ${tot:.2f}')
print(f'mean latency per rule set: '
      f'{np.mean([c["latency_seconds"] for c in costs.values()]):.1f}s')

MEAN AUTHORING COST (USD per rule set)

condition                       A2      A3      A4      A5
model_id                                                  
claude-haiku-4-5-20251001   0.0092  0.0156  0.0202  0.0279
claude-sonnet-4-5-20250929  0.0486  0.0543  0.0638  0.0821
gemini-flash                0.0000  0.0000  0.0000  0.0000


MEAN COST PER MILLION RETAINED TRAINING RECORDS (USD)

condition                                     A2     A3     A4     A5
corpus_id      model_id                                              
bank_marketing claude-haiku-4-5-20251001   0.574  0.373  0.503  0.628
               claude-sonnet-4-5-20250929  1.404  2.712  3.161  6.522
               gemini-flash                0.000  0.000  0.000  0.000
diabetes_130us claude-haiku-4-5-20251001     NaN  2.078  0.589  0.963
               claude-sonnet-4-5-20250929  4.454  9.854  2.433  2.184
               gemini-flash                0.000  0.000  0.000  0.000
nyc_tlc_yellow claude-haiku-4-5-20251001   0.078

## 12 · Save and provenance

In [ ]:
feas.to_csv(ARTIFACTS / 'N4_downstream.csv', index=False)
sub.to_csv(ARTIFACTS / 'N4_subgroups.csv', index=False)
res.to_csv(ARTIFACTS / 'N4_all_gates.csv', index=False)

prov = {
    'notebook': 'P2_N4_downstream',
    'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'downstream_version': DS.DOWNSTREAM_VERSION,
    'excluded_models': sorted(EXCLUDE_MODELS),
    'test_fraction': DS.TEST_FRACTION,
    'split_seed': DS.SPLIT_SEED,
    'tasks': {c: {'target': t.target_column, 'dropped': t.drop_columns,
                  'subgroups': t.subgroups, 'note': t.note}
              for c, t in DS.TASKS.items()},
    'baselines': {b['corpus_id']: {'roc_auc': b['roc_auc'], 'pr_auc': b['pr_auc'],
                                   'train_rows': b['train_rows']} for b in baselines},
    'n_gates': int(len(res)), 'n_infeasible': int((~res.feasible).sum()),
}
(ARTIFACTS / 'N4_provenance.json').write_text(json.dumps(prov, indent=2, default=str))
print('wrote', ARTIFACTS / 'N4_downstream.csv')
print('wrote', ARTIFACTS / 'N4_provenance.json')
print()
ckpt.status()

wrote /content/drive/MyDrive/Paper2_RuleAuthorship/artifacts/N4_downstream.csv
wrote /content/drive/MyDrive/Paper2_RuleAuthorship/artifacts/N4_provenance.json

Checkpoints in /content/drive/MyDrive/Paper2_RuleAuthorship/checkpoints
  OK  a3_doc_coverage.json                     708 B
  OK  census_bank_marketing.json               15,728 B
  OK  census_diabetes_130us.json               42,999 B
  OK  census_nyc_tlc_yellow.json               21,889 B
  OK  census_online_retail_ii.json             12,491 B
  OK  n0_provenance.json                       4,816 B
  OK  n1_provenance.json                       3,126 B
  OK  n3_rule_corpus.json                      4,695,903 B
  OK  n3_scored_rulesets.json                  243,773 B
  OK  n4_baselines.json                        3,785 B
  OK  n4_downstream_results.json               438,716 B
  OK  prompt_payloads.json                     176,012 B
  OK  source_file_hashes.json                  704 B
  OK  split_bank_marketing.json            

In [ ]:
import numpy as np, pandas as pd, json

CID = 'nyc_tlc_yellow'
ev  = prepared[CID]['ev']
print(f'evaluation split: {len(ev):,} rows\n')

# ---------------------------------------------------------------- Section A
print('=' * 66)
print('A. IS THE TRIPLE COINCIDENCE EXACT?')
print('=' * 66)

rate_null = ev['RatecodeID'].isna()
pay_zero  = (ev['payment_type'] == 0)
pass_null = ev['passenger_count'].isna()

print(f'RatecodeID IS NULL      : {rate_null.sum():>9,}  ({rate_null.mean():.4%})')
print(f'payment_type == 0       : {pay_zero.sum():>9,}  ({pay_zero.mean():.4%})')
print(f'passenger_count IS NULL : {pass_null.sum():>9,}  ({pass_null.mean():.4%})')

BLOCK = rate_null & pay_zero & pass_null
union = rate_null | pay_zero | pass_null
print(f'\nintersection of all three: {BLOCK.sum():>9,}')
print(f'union of all three       : {union.sum():>9,}')
print(f'Jaccard(intersection, union) = {BLOCK.sum() / max(union.sum(), 1):.6f}')

exact = (rate_null.sum() == pay_zero.sum() == pass_null.sum() == BLOCK.sum())
print(f'\n{"CONFIRMED" if exact else "NOT EXACT"}: the three conditions '
      f'{"identify identical row sets" if exact else "do NOT coincide exactly"}')
if not exact:
    print('  !! The manuscript claims an exact coincidence. Correct the claim')
    print('     to report the Jaccard above instead of asserting identity.')

# is the block predictive of the label?
y = prepared[CID]['y']
print(f'\nrecorded-tip rate inside block : {y[BLOCK.values].mean():.4f}')
print(f'recorded-tip rate outside block: {y[~BLOCK.values].mean():.4f}')
print('   (a large gap means the block is not a random slice — which is why')
print('    removing it damages the model rather than merely shrinking it)')

# ---------------------------------------------------------------- Section B
print('\n' + '=' * 66)
print('B. WHICH RULES REMOVE THE BLOCK?')
print('=' * 66)

seen, rows = set(), []
for key, rs in rulesets.items():
    if key[0] != CID:
        continue
    for rule in rs.rules:
        if not rule.is_executable or rule.column not in ev.columns:
            continue
        canon = IR.canonical_form(rule)
        if canon in seen:
            continue
        seen.add(canon)
        m = rule_mask(CID, rule)
        if m is None:
            continue
        rejected = ~m
        n_rej = int(rejected.sum())
        if n_rej == 0:
            continue
        inter = int((rejected & BLOCK).sum())
        rows.append({
            'column': rule.column,
            'predicate': rule.predicate_type.value,
            'params': json.dumps(rule.parameters)[:46],
            'rejects': n_rej,
            'pct_of_block_removed': inter / max(int(BLOCK.sum()), 1),
            'pct_of_rejections_in_block': inter / n_rej,
        })

ruledf = pd.DataFrame(rows).sort_values('pct_of_block_removed', ascending=False)
print(f'{len(ruledf)} distinct rules reject at least one record\n')
print('Rules removing the largest share of the block:')
print(ruledf.head(12).to_string(index=False,
      formatters={'pct_of_block_removed': '{:.1%}'.format,
                  'pct_of_rejections_in_block': '{:.1%}'.format,
                  'rejects': '{:,}'.format}))

pure = ruledf[(ruledf.pct_of_block_removed > 0.95) &
              (ruledf.pct_of_rejections_in_block > 0.95)]
print(f'\n{len(pure)} rules remove >95% of the block AND >95% of what they')
print('reject is inside it — these are block-removal rules in disguise.')

# ---------------------------------------------------------------- Section C
print('\n' + '=' * 66)
print('C. DOES BLOCK REMOVAL PREDICT DOWNSTREAM HARM?')
print('=' * 66)

train = prepared[CID]['train']
blk_train = BLOCK.reindex(train).fillna(False)

recs = []
for key, rs in rulesets.items():
    if key[0] != CID:
        continue
    keep = gate_mask(CID, rs).reindex(train).fillna(False)
    removed = ~keep
    recs.append({
        'condition': key[1], 'model': key[2], 'seed': key[3],
        'block_removed_frac': float((removed & blk_train).sum() /
                                    max(int(blk_train.sum()), 1)),
        'retention': float(keep.mean()),
    })

gates = pd.DataFrame(recs)
# Ensure delta_auc is present in 'feas' before merging.
# 'feas' is defined and 'delta_auc' is added in cell `lLDWkUUFZ1DS` which is earlier in the notebook execution flow.
d4 = feas[(feas.corpus_id == CID) & feas.feasible][
    ['condition', 'model_id', 'seed', 'delta_auc']].rename(
    columns={'model_id': 'model'})
merged = gates.merge(d4, on=['condition', 'model', 'seed'], how='inner')
print(f'{len(merged)} feasible C4 gates matched\n')

if len(merged) >= 5:
    r = np.corrcoef(merged.block_removed_frac, merged.delta_auc)[0, 1]
    print(f'correlation(block removed, delta AUC) = {r:+.4f}')
    hi = merged[merged.block_removed_frac > 0.5]
    lo = merged[merged.block_removed_frac <= 0.5]
    print(f'\n  gates removing >50% of block (n={len(hi)}): '
          f'mean delta AUC {hi.delta_auc.mean():+.4f}')
    print(f'  gates removing <=50%        (n={len(lo)}): '
          f'mean delta AUC {lo.delta_auc.mean():+.4f}')
    if len(hi) and len(lo):
        print(f'  difference: {hi.delta_auc.mean() - lo.delta_auc.mean():+.4f}')
    print('\n  A strong negative correlation supports the causal chain.')
    print('  A weak or positive one means the manuscript claim is unsupported')
    print('  and Figure 4 must be withdrawn or rewritten.')
    print('\nby condition:')
    print(merged.groupby('condition')[['block_removed_frac', 'delta_auc']]
          .mean().round(4).to_string())
else:
    print('too few matched gates to assess')

print('\n' + '=' * 66)
print('Paste this entire output back for the manuscript update.')
print('=' * 66)

evaluation split: 250,000 rows

A. IS THE TRIPLE COINCIDENCE EXACT?
RatecodeID IS NULL      :    58,621  (23.4484%)
payment_type == 0       :    58,621  (23.4484%)
passenger_count IS NULL :    58,621  (23.4484%)

intersection of all three:    58,621
union of all three       :    58,621
Jaccard(intersection, union) = 1.000000

CONFIRMED: the three conditions identify identical row sets

recorded-tip rate inside block : 0.0912
recorded-tip rate outside block: 0.7825
   (a large gap means the block is not a random slice — which is why
    removing it damages the model rather than merely shrinking it)

B. WHICH RULES REMOVE THE BLOCK?
86 distinct rules reject at least one record

Rules removing the largest share of the block:
              column predicate                              params rejects pct_of_block_removed pct_of_rejections_in_block
          RatecodeID    in_set     {"allowed": [1, 2, 3, 4, 5, 6]}  67,236               100.0%                      87.2%
     passenger_count  

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [ ]:
import numpy as np, pandas as pd

CID   = 'nyc_tlc_yellow'
p     = prepared[CID]
ev    = p['ev']
train = p['train']

rate_null = ev['RatecodeID'].isna()
pay_zero  = (ev['payment_type'] == 0)
pass_null = ev['passenger_count'].isna()
BLOCK     = rate_null & pay_zero & pass_null

blk_train = BLOCK.reindex(train).fillna(False)
n_block   = int(blk_train.sum())
print(f'train rows      : {len(train):,}')
print(f'block in train  : {n_block:,} ({n_block/len(train):.2%})\n')

rng = np.random.default_rng(20260807)

def mask_from_removed(removed_idx):
    """Keep-mask over the evaluation index given rows to remove."""
    keep = pd.Series(True, index=ev.index)
    keep.loc[removed_idx] = False
    return keep

# --- construct the four gates ---------------------------------------------
block_idx = train[blk_train.values]

rand_idx = pd.Index(rng.choice(train.values, size=n_block, replace=False))

outside = train[~blk_train.values]
comp_idx = pd.Index(rng.choice(outside.values,
                               size=min(n_block, len(outside)), replace=False))

gates = {
    'BASELINE       (no gate)':            None,
    'BLOCK_ONLY     (remove the block)':   mask_from_removed(block_idx),
    'RANDOM_MATCHED (same count, random)': mask_from_removed(rand_idx),
    'COMPLEMENT     (same count, outside)': mask_from_removed(comp_idx),
}

# --- fit and evaluate ------------------------------------------------------
print(f'{"gate":<38}{"train rows":>11}{"pos rate":>10}{"ROC-AUC":>10}{"delta":>9}')
print('-' * 78)

results, base_auc = {}, None
for name, keep in gates.items():
    r = DS.fit_and_evaluate(
        p['X'], p['y'], p['strata'], train, p['test'], keep,
        corpus_id=CID, condition=name.split()[0], model_id='counterfactual',
        seed=0)
    results[name] = r
    if base_auc is None:
        base_auc = r.roc_auc
    delta = r.roc_auc - base_auc
    if not r.feasible:
        print(f'{name:<38}{"INFEASIBLE":>11}  {r.reason}')
        continue
    print(f'{name:<38}{r.train_rows:>11,}{r.positive_rate_train:>10.4f}'
          f'{r.roc_auc:>10.4f}{delta:>+9.4f}')

# --- verdict ---------------------------------------------------------------
print('\n' + '=' * 78)
b = results['BLOCK_ONLY     (remove the block)']
m = results['RANDOM_MATCHED (same count, random)']
c = results['COMPLEMENT     (same count, outside)']

if b.feasible and m.feasible:
    d_block  = b.roc_auc - base_auc
    d_random = m.roc_auc - base_auc
    print(f'removing the block        : {d_block:+.4f} AUC')
    print(f'removing the same count   : {d_random:+.4f} AUC  (random rows)')
    if c.feasible:
        print(f'removing the same count   : {c.roc_auc - base_auc:+.4f} AUC  '
              '(rows outside the block)')
    gap = d_block - d_random
    print(f'\nattributable to WHICH rows: {gap:+.4f} AUC')
    print()
    if gap < -0.05:
        print('CAUSAL CLAIM SUPPORTED.')
        print('  Removing the block costs substantially more than removing the')
        print('  same number of arbitrary rows. The harm comes from the identity')
        print('  of the removed records, not the volume. Section 4.8 and')
        print('  Figure 4 stand as written; cite this contrast as the evidence.')
    elif gap < -0.01:
        print('CAUSAL CLAIM PARTIALLY SUPPORTED.')
        print('  The block-specific effect is real but modest. Soften Section')
        print('  4.8 to report both the volume and identity components rather')
        print('  than attributing the whole loss to the block.')
    else:
        print('CAUSAL CLAIM NOT SUPPORTED.')
        print('  Removing the block costs no more than removing an equal number')
        print('  of arbitrary rows. The degradation is driven by data VOLUME,')
        print('  not by the block. Rewrite Section 4.8 as an association and')
        print('  withdraw Figure 4 in its current form.')

print('\nlabel composition:')
y = p['y']
print(f'  recorded-tip rate inside block : {y[BLOCK.values].mean():.4f}')
print(f'  recorded-tip rate outside block: {y[~BLOCK.values].mean():.4f}')
print(f'  training positive rate, no gate     : '
      f'{results["BASELINE       (no gate)"].positive_rate_train:.4f}')
if b.feasible:
    print(f'  training positive rate, block removed: {b.positive_rate_train:.4f}')
print('=' * 78)

train rows      : 175,000
block in train  : 41,156 (23.52%)

gate                                   train rows  pos rate   ROC-AUC    delta
------------------------------------------------------------------------------
BASELINE       (no gate)                  175,000    0.6199    0.8687  +0.0000
BLOCK_ONLY     (remove the block)         133,844    0.7825    0.4931  -0.3755
RANDOM_MATCHED (same count, random)       133,844    0.6207    0.8683  -0.0004
COMPLEMENT     (same count, outside)      133,844    0.5694    0.8683  -0.0004

removing the block        : -0.3755 AUC
removing the same count   : -0.0004 AUC  (random rows)
removing the same count   : -0.0004 AUC  (rows outside the block)

attributable to WHICH rows: -0.3752 AUC

CAUSAL CLAIM SUPPORTED.
  Removing the block costs substantially more than removing the
  same number of arbitrary rows. The harm comes from the identity
  of the removed records, not the volume. Section 4.8 and
  Figure 4 stand as written; cite this contrast a